# Week 3 Exercises

## Epipolar Geometry

In [79]:
import numpy as np
from scipy.spatial.transform import Rotation
import sys
sys.path.append('..')
from CV_functions import box3d, Pi, PiInv, projectpoints

K = np.array([
    [1000, 0, 300],
    [0, 1000, 200],
    [0, 0, 1]
])

R1 = np.eye(3)
t1 = np.zeros((3, 1))
R2 = Rotation.from_euler('xyz', [0.7, -0.5, 0.8]).as_matrix()
t2 = np.array([[0.2], [2], [1]])

### Ex 3.1

In [80]:
Q = np.array([1, 0.5, 4]).reshape(3, 1)

q1 = projectpoints(K, R1, t1, Q)
q2 = projectpoints(K, R2, t2, Q)

print(q1, q2)

[[550.]
 [325.]] [[582.47256835]
 [185.98985776]]


### Ex 3.2

In [81]:
def CrossOp(p):
    p = p.flatten()
    return np.cross(np.eye(3), p)

p = np.array([1, 2, 3]).reshape(3, 1)
CrossOp(p)

array([[ 0., -3.,  2.],
       [ 3.,  0., -1.],
       [-2.,  1.,  0.]])

### Ex 3.3

In [82]:
E1 = CrossOp(t1) @ R1
E2 = CrossOp(t2) @ R2

F = np.linalg.inv(K.T) @ E2 @ np.linalg.inv(K)

print(F)

[[ 3.29311881e-07  8.19396327e-07  1.79162592e-03]
 [ 5.15532551e-07 -8.76915984e-07  9.31426656e-05]
 [-1.29882755e-03  1.51951700e-03 -1.10072682e+00]]


### Ex 3.4

In [83]:
l = F @ PiInv(q1)

print(l)

[[ 2.23905126e-03]
 [ 9.16878739e-05]
 [-1.32123895e+00]]


### Ex 3.5

In [84]:
PiInv(q2).T @ l

array([[4.30211422e-16]])

Basically 0 (e-16) somust be on the line

### Ex 3.8

In [85]:
ims = np.load('TwoImageDataCar.npy', allow_pickle=True).item()
print(ims.keys())

dict_keys(['im1', 'R1', 't1', 'im2', 'R2', 't2', 'K'])


### Ex 3.9

In [86]:
import matplotlib.pyplot as plt

def DrawLine(l, shape):
    #Checks where the line intersects the four sides of the image
    # and finds the two intersections that are within the frame
    def in_frame(l_im):
        q = np.cross(l.flatten(), l_im)
        q = q[:2]/q[2]
        if all(q>=0) and all(q+1<=shape[1::-1]):
            return q
    lines = [[1, 0, 0], [0, 1, 0], [1, 0, 1-shape[1]], [0, 1, 1-shape[0]]]
    P = [in_frame(l_im) for l_im in lines if in_frame(l_im) is not None]
    if (len(P)==0):
        print("Line is completely outside image")
    plt.plot(*np.array(P).T)

%matplotlib qt

fig, (ax1, ax2) = plt.subplots(1, 2)
ax1.imshow(ims['im1'])
ax2.imshow(ims['im2'])

q1 = plt.ginput(1)[0]        # click on image 1
q1 = np.array(q1).reshape(2, 1)

l = F @ PiInv(q1)            # epipolar line in image 2

ax1.plot(*q1, 'ro')
plt.sca(ax2)
DrawLine(l, ims['im2'].shape)

plt.show()

### Ex 3.10

In [87]:
fig, (ax1, ax2) = plt.subplots(1, 2)
ax1.imshow(ims['im1'])
ax2.imshow(ims['im2'])

q2 = plt.ginput(1)[0]        # click on image 2
q2 = np.array(q2).reshape(2, 1)

l = F.T @ PiInv(q2)          # epipolar line in image 1

ax2.plot(*q2.flatten(), 'ro')
plt.sca(ax1)
DrawLine(l, ims['im1'].shape)

plt.show()

### Ex 3.11

In [88]:
def triangulate(qs, Ps):
    """
    qs: list of n pixel coords, each shape (2,) or (2,1) — inhomogeneous
    Ps: list of n projection matrices, each shape (3,4)
    Returns: Q, the triangulated 3D point (homogeneous, shape (4,1))
    """
    B = []
    for q, P in zip(qs, Ps):
        q = np.array(q).flatten()
        x, y = q[0], q[1]
        B.append(x * P[2] - P[0])
        B.append(y * P[2] - P[1])
    B = np.array(B)

    U, S, Vt = np.linalg.svd(B)
    Q = Vt[-1]          # smallest singular vector = null-space solution
    Q = Q / Q[-1]       # normalize so last coord is 1
    return Q.reshape(-1, 1)